In [1]:
# CELL 1: Environment setup
import json
import numpy as np
import pandas as pd
import chromadb
from pathlib import Path
from chromadb.utils import embedding_functions
from llama_cpp import Llama, LlamaGrammar

RESULTS_DIR  = Path("../data/results")
ATTCK_DIR    = Path("../data/attck")
CHROMA_DIR   = Path("../data/chroma")
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH   = "../models/qwen2.5-3b-instruct-q4_k_m.gguf"

COMMUNITY_FILE = RESULTS_DIR / "community_assignments.csv"
TRIPLES_FILE   = RESULTS_DIR / "community_triples.json"
STIX_FILE      = ATTCK_DIR   / "enterprise-attack.json"
GRAPH_NODES    = RESULTS_DIR / "knowledge_graph_nodes.csv"
GRAPH_EDGES    = RESULTS_DIR / "knowledge_graph_edges.csv"

REPORTS_FILE   = RESULTS_DIR / "stage5_rag_reports.csv"
METRICS_FILE   = RESULTS_DIR / "stage5_rag_metrics.json"

EMBED_MODEL  = "all-MiniLM-L6-v2"  # Same model used in Stage 2 — consistent semantic space
TOP_K        = 5                    # Retrieve top-5 ATT&CK techniques per community
RANDOM_SEED  = 42

print("Stage 5 environment ready.")

Stage 5 environment ready.


In [2]:
# CELL 2: Load all Stage 3 and Stage 4 outputs
community_df = pd.read_csv(COMMUNITY_FILE, low_memory=False)

with open(TRIPLES_FILE, "r", encoding="utf-8") as f:
    community_triples = json.load(f)

# Load knowledge graph edges for provenance
kg_edges = pd.read_csv(GRAPH_EDGES)
kg_nodes = pd.read_csv(GRAPH_NODES)

# Only evaluate attack communities (not benign)
eval_df  = community_df[community_df["attck_technique_id"] != "BENIGN"].copy()
eval_cids = sorted(eval_df["community_id"].unique())

print(f"Loaded {len(community_df)} alerts across {community_df['community_id'].nunique()} communities")
print(f"Attack communities to evaluate: {len(eval_cids)}")
print(f"Knowledge graph: {len(kg_nodes)} nodes, {len(kg_edges)} edges")

Loaded 10684 alerts across 19 communities
Attack communities to evaluate: 14
Knowledge graph: 38 nodes, 43 edges


In [3]:
# CELL 3: Build ChromaDB vector store from ATT&CK STIX bundle
#
# The same embedding model (all-MiniLM-L6-v2) is used here as in Stage 2.
# This ensures the alert-derived queries and the ATT&CK technique descriptions
# live in the same semantic vector space, making similarity comparisons valid.

ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(name="attack_techniques", embedding_function=ef)

if collection.count() == 0:
    print("Populating ChromaDB with ATT&CK techniques...")
    with open(STIX_FILE, "r", encoding="utf-8") as f:
        bundle = json.load(f)

    docs, ids, metas = [], [], []
    for obj in bundle.get("objects", []):
        if obj.get("type") != "attack-pattern" or obj.get("revoked"):
            continue
        tid = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                tid = ref.get("external_id")
                break
        if not tid:
            continue

        tactics = [
            p["phase_name"].replace("-", " ").title()
            for p in obj.get("kill_chain_phases", [])
            if p.get("kill_chain_name") == "mitre-attack"
        ]
        # Store full text: ID + name + tactic + description
        # The description is what the embedding model reads to find semantically similar techniques
        text = (
            f"ID: {tid}\n"
            f"Name: {obj.get('name', '')}\n"
            f"Tactic: {', '.join(tactics)}\n"
            f"Description: {obj.get('description', '')}"
        )
        docs.append(text)
        ids.append(tid)
        metas.append({"technique_id": tid, "name": obj.get("name", ""), "tactic": ", ".join(tactics)})

    collection.add(ids=ids, documents=docs, metadatas=metas)
    print(f"Inserted {collection.count()} ATT&CK technique descriptions")
else:
    print(f"ChromaDB already populated: {collection.count()} technique descriptions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB already populated: 703 technique descriptions


In [4]:
# CELL 4: Query builder — this is where the knowledge graph earns its place
#
# The original version built the retrieval query only from port numbers and
# the enum-constrained triples. Because those triples were meaningless (all
# "network_flow -> something"), the query carried almost no signal.
#
# With the fixed Stage 3, triples contain real extracted entities.
# The query is now built from three layers:
#   1. Raw alert behavior (from alert_text samples)
#   2. Knowledge graph triples for this community (semantic relationships)
#   3. Graph-level context: recurring edges from kg_edges that involve
#      the same entities — this adds cross-community signal
#
# This is exactly the professor's intended flow:
# alert text -> embeddings -> communities -> LLM extracts triples ->
# KG built -> KG-derived query -> ChromaDB retrieval -> grounded report

def build_rag_query(cid: int) -> str:
    """Build a rich retrieval query for community `cid` using the knowledge graph."""
    group = community_df[community_df["community_id"] == cid]
    triples = community_triples.get(str(cid), [])

    # Layer 1: Alert text samples (raw observed behavior)
    raw_alerts = group["alert_text"].dropna().head(3).tolist()
    alert_snippet = " ".join(raw_alerts)

    # Layer 2: Knowledge graph triples as natural language sentences
    # Convert "ssh_scanner --[targets]--> port_22_service" to
    # "ssh_scanner targets port_22_service"
    triple_sentences = []
    for t in triples:
        subj = t.get("subject", "").replace("_", " ")
        rel  = t.get("relation", "").replace("_", " ")
        tgt  = t.get("target", "").replace("_", " ")
        if subj and rel and tgt:
            triple_sentences.append(f"{subj} {rel} {tgt}")
    triple_text = ". ".join(triple_sentences)

    # Layer 3: High-weight edges from the knowledge graph that share entities
    # with this community's triples — adds cross-community corroboration
    community_entities = set()
    for t in triples:
        community_entities.add(t.get("subject", ""))
        community_entities.add(t.get("target", ""))

    related_edges = kg_edges[
        kg_edges["source"].isin(community_entities) |
        kg_edges["target"].isin(community_entities)
    ].sort_values("weight", ascending=False).head(3)

    graph_context = ""
    if not related_edges.empty:
        parts = [
            f"{row['source'].replace('_',' ')} {row['relation'].replace('_',' ')} {row['target'].replace('_',' ')}"
            for _, row in related_edges.iterrows()
        ]
        graph_context = "Recurring graph patterns: " + "; ".join(parts) + "."

    # Combine all layers into a single retrieval query string
    query = f"{triple_text}. {graph_context} {alert_snippet}".strip()
    return query


# Verify with one community
sample_cid = eval_cids[0]
print(f"Example RAG query for community {sample_cid}:")
print(build_rag_query(sample_cid))

Example RAG query for community 0:
ftp client targets ftp port 21 service. ftp client scans http web server. http flood source floods http web server. http flood source targets http port 80 service. Recurring graph patterns: dns resolver scans http web server; ftp client floods network bandwidth; ftp client scans ftp port 21 service. Flow to HTTP. Duration 5382184us. Fwd pkts 3, Bwd pkts 1. Bytes/s 0.0. Flags none. Behavior: long_duration. Flow to HTTP-alt. Duration 68920us. Fwd pkts 4, Bwd pkts 3. Bytes/s 4933.26. Flags none. Behavior: observed_pattern. Flow to HTTP-alt. Duration 997272us. Fwd pkts 3, Bwd pkts 3. Bytes/s 18.05. Flags none. Behavior: observed_pattern.


In [5]:
# CELL 5: LLM and report grammar
#
# Qwen2.5-3B is used here for report generation (smaller than Stage 3 extraction).
# For report generation the task is simpler: given context, produce structured output.
# The 3B model handles this well with grammar constraints and a clear prompt.
#
# n_gpu_layers=-1 means load all layers onto GPU.
# At Q4_K_M the 3B model is ~2.0GB — fits comfortably within 4GB VRAM.
#
# The report schema captures the four fields your thesis evaluation needs:
# - technique_id: the ATT&CK ID the model selects (grounding check)
# - tactic: the broad ATT&CK tactic category
# - summary: a human-readable description of what was observed
# - evidence: specific observations from the alert data that support the classification
# - next_step: recommended analyst action (makes the output "actionable intelligence")

REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "technique_id": {"type": "string"},
        "tactic":        {"type": "string"},
        "summary":       {"type": "string"},
        "evidence":      {"type": "string"},
        "next_step":     {"type": "string"}
    },
    "required": ["technique_id", "tactic", "summary", "evidence", "next_step"]
}
report_grammar = LlamaGrammar.from_json_schema(json.dumps(REPORT_SCHEMA))

print(f"Loading {MODEL_PATH}...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,
    n_gpu_layers=-1,   # Full GPU offload — 3B Q4_K_M fits in 4GB VRAM
    n_threads=8,
    n_batch=256,
    verbose=False,
    seed=RANDOM_SEED,
)
print("LLM ready.")

Loading ../models/qwen2.5-3b-instruct-q4_k_m.gguf...


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


LLM ready.


In [6]:
# CELL 6: Report generation functions — RAG and Baseline
#
# Two configurations are run for every community so you can compare them:
#
# RAG mode: query -> ChromaDB retrieval -> LLM reads retrieved ATT&CK context
# Baseline mode: LLM generates without any retrieved context
#
# This comparison directly answers your thesis hypothesis:
# "Does knowledge graph-grounded retrieval improve report quality over LLM-only generation?"
# The metrics (grounding_rate, exact_match_rate, parent_match_rate) measure the answer.

MAX_CHARS_PER_DOC = 400  # Keep context within token budget for 3B model


def generate_rag_report(query: str, retrieved_docs: list, retrieved_metas: list) -> dict:
    """Generate a report grounded in retrieved ATT&CK context."""
    allowed_ids = [m.get("technique_id", "") for m in retrieved_metas]
    allowed_str = ", ".join(allowed_ids)

    # Truncate each retrieved doc to stay within context window
    context_block = "\n\n".join([
        f"[{i+1}] {str(doc)[:MAX_CHARS_PER_DOC]}"
        for i, doc in enumerate(retrieved_docs)
    ])

    prompt = f"""[INST] You are a senior cybersecurity analyst writing an incident report.

You MUST choose technique_id from this list only: {allowed_str}

OBSERVED NETWORK BEHAVIOR:
{query}

RETRIEVED ATT&CK TECHNIQUES:
{context_block}

Based on the observed behavior and the retrieved techniques, write a structured incident report.
Return ONLY valid JSON. [/INST]"""

    out = llm(
        prompt, max_tokens=400, temperature=0,
        seed=RANDOM_SEED, grammar=report_grammar, repeat_penalty=1.1
    )
    raw = out["choices"][0]["text"].strip()

    try:
        result = json.loads(raw)
        # Grounding check: reject any technique_id not in the retrieved set
        if result.get("technique_id", "") not in allowed_ids:
            result["technique_id"] = "Unknown"
            result["evidence"] = "[Hallucinated ID corrected — not in retrieved context]"
        return result
    except json.JSONDecodeError:
        return {"technique_id": "Unknown", "tactic": "Unknown",
                "summary": "Parse error", "evidence": "N/A", "next_step": "Manual review"}


def generate_baseline_report(query: str) -> dict:
    """Generate a report with no retrieved context — LLM knowledge only."""
    prompt = f"""[INST] You are a senior cybersecurity analyst writing an incident report.

OBSERVED NETWORK BEHAVIOR:
{query}

Based only on the observed behavior, identify the most likely MITRE ATT&CK technique
and write a structured incident report.
Return ONLY valid JSON. [/INST]"""

    out = llm(
        prompt, max_tokens=400, temperature=0,
        seed=RANDOM_SEED, grammar=report_grammar, repeat_penalty=1.1
    )
    raw = out["choices"][0]["text"].strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"technique_id": "Unknown", "tactic": "Unknown",
                "summary": "Parse error", "evidence": "N/A", "next_step": "Manual review"}

In [7]:
# CELL 7: Evaluation loop — runs both RAG and Baseline for every attack community

def parent_match(gt_id: str, gen_id: str) -> bool:
    """True if generated ID matches the parent technique (e.g. T1110 for T1110.001)."""
    if not gt_id or not gen_id or gen_id == "Unknown":
        return False
    return gt_id.split(".")[0] == gen_id.split(".")[0]


print("Warming up LLM...")
_ = llm("[INST] Test. [/INST]", max_tokens=5, temperature=0, seed=RANDOM_SEED)
print("Warmup complete. Starting evaluation...\n")

results = []

for cid in eval_cids:
    group = community_df[community_df["community_id"] == cid]
    ground_truth = group["attck_technique_id"].mode().iloc[0]

    # Build the knowledge-graph-enriched query
    query = build_rag_query(cid)

    # --- RAG path ---
    retrieval = collection.query(query_texts=[query], n_results=TOP_K)
    docs    = retrieval["documents"][0]
    metas   = retrieval["metadatas"][0]
    retrieved_ids = [m.get("technique_id", "") for m in metas]

    rag_report = generate_rag_report(query, docs, metas)
    rag_id     = rag_report.get("technique_id", "Unknown")

    # --- Baseline path (same query, no retrieved context) ---
    base_report = generate_baseline_report(query)
    base_id     = base_report.get("technique_id", "Unknown")

    # --- Scoring ---
    retrieval_hit   = ground_truth in retrieved_ids
    rag_grounded    = rag_id in retrieved_ids or rag_id == "Unknown"
    rag_exact       = (rag_id == ground_truth) and rag_grounded
    rag_parent      = parent_match(ground_truth, rag_id) and rag_grounded
    base_exact      = base_id == ground_truth
    base_parent     = parent_match(ground_truth, base_id)

    results.append({
        "community_id":           cid,
        "ground_truth":           ground_truth,
        "dominant_label":         group["Label"].mode().iloc[0],
        "dominant_tactic":        group["attck_tactic"].mode().iloc[0],
        # Retrieval quality
        "retrieved_ids":          retrieved_ids,
        "retrieval_hit":          retrieval_hit,
        # RAG report
        "rag_technique_id":       rag_id,
        "rag_grounded":           rag_grounded,
        "rag_exact_match":        rag_exact,
        "rag_parent_match":       rag_parent,
        "rag_tactic":             rag_report.get("tactic", ""),
        "rag_summary":            rag_report.get("summary", ""),
        "rag_evidence":           rag_report.get("evidence", ""),
        "rag_next_step":          rag_report.get("next_step", ""),
        # Baseline report
        "base_technique_id":      base_id,
        "base_exact_match":       base_exact,
        "base_parent_match":      base_parent,
        "base_tactic":            base_report.get("tactic", ""),
        "base_summary":           base_report.get("summary", ""),
    })

    print(f"  Community {cid} [{ground_truth}] | "
          f"RAG: {rag_id} (exact={rag_exact}, parent={rag_parent}) | "
          f"Baseline: {base_id} (exact={base_exact}, parent={base_parent})")

print(f"\nEvaluation complete: {len(results)} communities")

Warming up LLM...
Warmup complete. Starting evaluation...

  Community 0 [T1046] | RAG: T1499.002 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 1 [T1110.001] | RAG: T1205.001 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 2 [BENIGN] | RAG: T1071.004 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 4 [T1110.001] | RAG: T1499.003 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 5 [T1110.001] | RAG: T1110.003 (exact=False, parent=True) | Baseline: T1078 (exact=False, parent=False)
  Community 6 [BENIGN] | RAG: T1205 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 10 [BENIGN] | RAG: T1071.004 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 11 [T1110.001] | RAG: T1499.002 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 12 [T1071

In [8]:
# CELL 8: Compute and print comparative metrics
#
# These metrics directly measure your thesis hypothesis.
# Every metric is computed for both RAG and Baseline so you can compare.
#
# retrieval_hit_rate:     How often does the correct ATT&CK ID appear in the top-K results?
#                         This measures ChromaDB retrieval quality independently.
# grounding_rate:         How often does the RAG model stay within the retrieved context?
#                         1.0 = never hallucinates outside the retrieved set.
# exact_match_rate:       How often does the generated ID exactly match ground truth?
# parent_match_rate:      How often does the generated ID match at parent technique level?
#                         (e.g. T1110 matches T1110.001) — more lenient, useful for sub-techniques.

n = len(results)

metrics = {
    "model":                     "qwen2.5-3b-instruct-q4_k_m",
    "n_communities_evaluated":   n,
    "top_k_retrieval":           TOP_K,
    "retrieval_hit_rate":        round(sum(r["retrieval_hit"]    for r in results) / n, 4),
    "rag_grounding_rate":        round(sum(r["rag_grounded"]     for r in results) / n, 4),
    "rag_exact_match_rate":      round(sum(r["rag_exact_match"]  for r in results) / n, 4),
    "rag_parent_match_rate":     round(sum(r["rag_parent_match"] for r in results) / n, 4),
    "baseline_exact_match_rate": round(sum(r["base_exact_match"] for r in results) / n, 4),
    "baseline_parent_match_rate":round(sum(r["base_parent_match"]for r in results) / n, 4),
}

print("=== STAGE 5 METRICS ===")
print(json.dumps(metrics, indent=2))

print("\n=== COMPARISON SUMMARY ===")
print(f"  RAG exact match:      {metrics['rag_exact_match_rate']:.2%}")
print(f"  Baseline exact match: {metrics['baseline_exact_match_rate']:.2%}")
delta_exact = metrics['rag_exact_match_rate'] - metrics['baseline_exact_match_rate']
print(f"  Delta (RAG - Baseline): {delta_exact:+.2%}")

print(f"\n  RAG parent match:      {metrics['rag_parent_match_rate']:.2%}")
print(f"  Baseline parent match: {metrics['baseline_parent_match_rate']:.2%}")
delta_parent = metrics['rag_parent_match_rate'] - metrics['baseline_parent_match_rate']
print(f"  Delta (RAG - Baseline): {delta_parent:+.2%}")

=== STAGE 5 METRICS ===
{
  "model": "qwen2.5-3b-instruct-q4_k_m",
  "n_communities_evaluated": 14,
  "top_k_retrieval": 5,
  "retrieval_hit_rate": 0.1429,
  "rag_grounding_rate": 1.0,
  "rag_exact_match_rate": 0.0,
  "rag_parent_match_rate": 0.0714,
  "baseline_exact_match_rate": 0.0,
  "baseline_parent_match_rate": 0.0
}

=== COMPARISON SUMMARY ===
  RAG exact match:      0.00%
  Baseline exact match: 0.00%
  Delta (RAG - Baseline): +0.00%

  RAG parent match:      7.14%
  Baseline parent match: 0.00%
  Delta (RAG - Baseline): +7.14%


In [9]:
# CELL 9: Inspect sample reports side by side
print("=== SAMPLE REPORTS: RAG vs BASELINE (First 3 Attack Communities) ===\n")
for r in results[:3]:
    print(f"Community {r['community_id']} | Label: {r['dominant_label']} | Tactic: {r['dominant_tactic']}")
    print(f"  Ground truth technique: {r['ground_truth']}")
    print(f"  Retrieved IDs:          {r['retrieved_ids']}")
    print(f"  " + "-"*50)
    print(f"  [RAG]      {r['rag_technique_id']} | exact={r['rag_exact_match']} parent={r['rag_parent_match']}")
    print(f"             Summary:  {r['rag_summary']}")
    print(f"             Evidence: {r['rag_evidence']}")
    print(f"             Action:   {r['rag_next_step']}")
    print(f"  [Baseline] {r['base_technique_id']} | exact={r['base_exact_match']} parent={r['base_parent_match']}")
    print(f"             Summary:  {r['base_summary']}")
    print("=" * 60 + "\n")

=== SAMPLE REPORTS: RAG vs BASELINE (First 3 Attack Communities) ===

Community 0 | Label: PortScan | Tactic: Impact
  Ground truth technique: T1046
  Retrieved IDs:          ['T1499.002', 'T1499.003', 'T1499.001', 'T1016.001', 'T1071.002']
  --------------------------------------------------
  [RAG]      T1499.002 | exact=False parent=False
             Summary:  The observed network behavior indicates a service exhaustion flood attack targeting the HTTP web server and FTP port 21 services, resulting in denial of service (DoS) conditions. The recurring patterns include DNS resolver scans, FTP client floods, and HTTP flood sources scanning HTTP ports.
             Evidence: [1] DNS resolver scans indicate adversaries are probing for Internet connectivity on compromised systems.
[2] FTP client floods suggest a targeted attack on the availability of FTP services.
[3] HTTP flood sources scanning HTTP ports imply an attempt to exhaust system resources by repeatedly requesting specific feat

In [10]:
# CELL 10: Save all outputs
results_df = pd.DataFrame(results)
results_df.to_csv(REPORTS_FILE, index=False)

with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved reports  -> {REPORTS_FILE}")
print(f"Saved metrics  -> {METRICS_FILE}")
print("\nStage 5 complete. Pipeline finished.")

Saved reports  -> ../data/results/stage5_rag_reports.csv
Saved metrics  -> ../data/results/stage5_rag_metrics.json

Stage 5 complete. Pipeline finished.
